# I/O benchmark for large MuData (backed modes only)

Benchmark **backed mode** read performance for large MuData files.
Compares lustre network reads vs local `/tmp` copy.

**Input**: `gastrula_to_pup_filtered.h5mu`

In [ ]:
import os
import shutil
import time

import mudata as mu
import numpy as np
import psutil

DATA_DIR = "/nfs/team205/vk7/sanger_projects/large_data/gastrula_to_pup"
h5mu_filename = "gastrula_to_pup_filtered_qc.h5mu"

In [ ]:
h5mu_filt = os.path.join(DATA_DIR, h5mu_filename)
N_READS = 100

# Get n_cells from backed read (no RAM needed)
mdata_lustre = mu.read_h5mu(h5mu_filt, backed="r")
n_cells = mdata_lustre["rna"].n_obs
batch_idx = np.sort(np.random.choice(n_cells, 1024, replace=False))
print(f"File: {h5mu_filt}")
print(f"RSS after backed open: {psutil.Process().memory_info().rss / 1e9:.1f} GB")
for mod_name in mdata_lustre.mod:
    print(f"  {mod_name}: {mdata_lustre[mod_name].shape}")

# Option B: Backed from lustre
t0 = time.time()
for _ in range(N_READS):
    for mod_name in mdata_lustre.mod:
        _ = mdata_lustre[mod_name].X[batch_idx]
t_lustre = time.time() - t0
mdata_lustre.file.close()
del mdata_lustre

# Option C: Backed from local /tmp
local_path = f"/tmp/{h5mu_filename}"
print(f"\nCopying to {local_path}...")
t_copy_start = time.time()
shutil.copy2(h5mu_filt, local_path)
t_copy = time.time() - t_copy_start
print(f"Copy took {t_copy:.0f}s ({os.path.getsize(h5mu_filt) / 1e9 / t_copy:.1f} GB/s)")
mdata_local = mu.read_h5mu(local_path, backed="r")
t0 = time.time()
for _ in range(N_READS):
    for mod_name in mdata_local.mod:
        _ = mdata_local[mod_name].X[batch_idx]
t_local = time.time() - t0
mdata_local.file.close()
del mdata_local
os.remove(local_path)

# Results
print(f"\n{'Option':<25} {'Time (s)':>10} {'Relative':>10}")
print("-" * 47)
print(f"{'Backed (lustre)':<25} {t_lustre:10.2f} {'1.0x':>10}")
print(f"{'Backed (/tmp local)':<25} {t_local:10.2f} {f'{t_local / t_lustre:.2f}x':>10}")

# Estimate full training time impact (25 epochs x ~N steps/epoch)
steps_per_epoch = n_cells // 1024
total_reads = 25 * steps_per_epoch
print(f"\nEstimated total training I/O overhead ({total_reads:,} reads):")
print(f"  Backed (lustre): {total_reads * t_lustre / N_READS / 3600:.1f} hours")
print(f"  Backed (/tmp): {total_reads * t_local / N_READS / 3600:.1f} hours")
print(f"  /tmp copy overhead: {t_copy / 60:.1f} min (one-time)")

## Recommendation

- Copy to `/tmp` and use **backed from /tmp** for best backed-mode performance.
- Avoid **backed from lustre** unless no other option — network I/O adds significant overhead.

Set the loading option in NB2 cell 3 accordingly.